In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from yellowbrick.cluster import KElbowVisualizer, SilhouetteVisualizer
from kneed import KneeLocator
import matplotlib.pyplot as plt
from IPython.display import display
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
plt.rcParams['font.sans-serif'] = ['Arial']
plt.style.use("seaborn-v0_8-talk")
%matplotlib inline

<div class="jumbotron">
    <h1 class="display-1">Clustering Advanced</h1>
    <hr class="my-4">
    <p>Instructor: Dr. Yan Li</p>
</div>


## The Business Question

<div class="alert alert-info">
    <strong>Setting:</strong> A hotel’s marketing team segmented guests with K-Means (Clustering I) using
    stay duration and daily spend. But real guests do not fall into clean spheres.
</div>

<dl class="row">
    <dt class="col-md-4 text-primary">Irregular segments</dt>
    <dd class="col-md-8">Segments can have <strong>any shape</strong>, not just spherical — K-Means splits them incorrectly.</dd>
    <dt class="col-md-4 text-warning">Unusual guests</dt>
    <dd class="col-md-8">Some guests behave abnormally (potential VIPs, group bookings, or abuse) — K-Means forces them into a segment instead of flagging them.</dd>
    <dt class="col-md-4 text-danger">Overlapping segments</dt>
    <dd class="col-md-8">A guest may sit between two segments — a hard label loses that uncertainty.</dd>
</dl>

**Goal:** find segments of <em>any shape</em>, <em>flag unusual guests</em>, and assign guests to segments <em>with confidence</em>.


### Roadmap — Advanced Clustering for Hotel Analytics

| Method | What it adds over Clustering I | Business value |
|--------|--------------------------------|----------------|
| **DBSCAN** (review) | Clusters of any shape + noise points | Flag unusual guests |
| **HDBSCAN** | Handles varying density; no `eps` to tune | Find mixed-density segments |
| **GMM** | Soft (probabilistic) assignment | Confidence + overlapping segments |


## Density-Based Clustering


### Basic Concepts


> Density-Based Spatial Clustering of Applications with Noise (DBSCAN)


<dl class="row alert-danger">
    <dt class="col-xl-4">Density</dt>
    <dd class="col-xl-8">The number of data points within a given radius ($Eps$).</dd>
</dl>


##### Point Classification


<dl class="row alert-danger">
    <dt class="col-xl-4">Core point</dt>
    <dd class="col-xl-8">Its $Eps$-neighborhood contains more than a threshold ($MinPts$) of points.</dd>
</dl>


<dl class="row alert-info">
    <dt class="col-xl-4">Border point</dt>
    <dd class="col-xl-8">Not a core point, but lies inside the $Eps$-neighborhood of a core point.</dd>
</dl>


<dl class="row alert-success">
    <dt class="col-xl-4">Noise point</dt>
    <dd class="col-xl-8">Neither a core point nor a border point.</dd>
</dl>


<center><img src="./img/clustering/densityPoints.gif" width=100%></center>

### Basic Algorithm


1. Label every point as a core point, border point, or noise point
2. Remove noise points
3. Connect all core points that lie within distance $Eps$ of each other
4. Each connected group of core points forms a cluster
5. Assign each border point to the cluster of an associated core point


<center><img src="./img/clustering/dbscan_algorithm_steps.png" width="90%"></center>


### Advantages


- Unlike $K$-means, it does not assume a specific cluster shape


- Unlike $K$-means and hierarchical clustering, it does not force every point into a cluster — noise points can be removed


### DBSCAN in Python


```python
from sklearn.cluster import DBSCAN
DBSCAN(eps=0.5, min_samples=5, metric='euclidean', n_jobs=None)
```
- `eps`: `float`, the radius; the maximum distance for two points to be considered in the same neighborhood
- `min_samples`: `int`, the minimum number of points (including the core point) a core point neighborhood must contain
- `metric`: the distance metric, e.g. `'minkowski'`, `'euclidean'`


#### Build the DBSCAN Model


In [ ]:
# --- Load and standardize the hotel guest features ---
hotdf = pd.read_csv('data/analysis/hotel_bookings.csv')
hotdf['Stay_Duration_Days'] = hotdf['stays_in_weekend_nights'] + hotdf['stays_in_week_nights']
hotdf['Daily_Spend_ADR'] = hotdf['adr']
hotdf['Total_Spend'] = hotdf['Stay_Duration_Days'] * hotdf['Daily_Spend_ADR']
hotdf = hotdf[hotdf['Total_Spend'] > 0] # Filter out zero spend
from sklearn.preprocessing import StandardScaler

scaled_df = pd.DataFrame(
    StandardScaler().fit_transform(hotdf[['Stay_Duration_Days', 'Daily_Spend_ADR']]),
    columns=['Stay_Duration_Days', 'Daily_Spend_ADR'])
scaled_df.head()

In [ ]:
from sklearn.cluster import DBSCAN
hotDB = DBSCAN(eps=0.5, min_samples=30, metric='euclidean', n_jobs=-1)  # how eps is chosen is shown below


#### Fit the Model and Predict Cluster Labels


In [ ]:
hotLabelsDB = hotDB.fit_predict(scaled_df)
pd.Series(hotLabelsDB).value_counts().sort_index()

- $-1$ marks the noise points


#### Choosing `min_samples`


- `min_samples` too small → too many clusters
- `min_samples` too large → too few clusters


- Rule of thumb: $\text{min\_samples}=2\times\text{n\_features}$, or $\text{min\_samples}=\text{n\_features}+1$
- Increase `min_samples` for very large datasets (denser point sampling supports larger neighborhoods)
- Larger `min_samples` makes the density criterion stricter, which suppresses spurious micro-clusters in noisy data (at the cost of discarding more points)


#### Choosing the `eps` Radius


- Use the $k$-nearest-neighbor distances to gauge the scale of the data
- Set $k=\text{min\_samples}$
- Plot the sorted $k$-NN distances; the distance at the **elbow** of the curve is `eps`


```python
from sklearn.neighbors import NearestNeighbors
NearestNeighbors(n_neighbors=k, n_jobs=None)  # k = min_samples
```
- `n_neighbors`: the number of neighbors
- `n_jobs`: how many CPU cores to use in parallel; -1 means all cores


##### Failure Case: The Real Hotel Data


The real hotel guests have **no such clean gap**. Apply the exact same steps and watch the knee method break down.


In [ ]:
# --- k-NN distances on the real (standardized) hotel data ---
nbrsHot = NearestNeighbors(n_neighbors=4)
nbrsHot.fit(scaled_df)
distancesHot = np.sort(nbrsHot.kneighbors(scaled_df)[0][:, -1], axis=0)


In [ ]:
kneeHot = KneeLocator(range(len(distancesHot)),distancesHot,S=1,curve='convex',direction='increasing',online=False)
kneeHot.plot_knee() # 绘制曲线与拐点位置
distancesHot[kneeHot.elbow]

##### The Knee Method Fails Here


In [ ]:
# --- The knee locator returns a tiny eps that over-fragments the data ---
kneeHot = KneeLocator(range(len(distancesHot)), distancesHot, S=1,
                      curve='convex', direction='increasing', online=False)
eps_bad = distancesHot[kneeHot.elbow]
print(f'knee eps = {eps_bad:.3f}')
badDB = DBSCAN(eps=eps_bad, min_samples=30, metric='euclidean', n_jobs=-1)
badLabels = badDB.fit_predict(scaled_df)
pd.Series(badLabels).value_counts().sort_index()

The knee returns `eps ≈ 0.027`, deep inside each dense cloud. Real segments are **density-stratified** — a dense core with a sparse halo — so a tiny `eps` splits every segment into several micro-clusters: 6 "segments". The sorted-distance curve has no single clear elbow, so the knee method is unreliable here.

This is a key **DBSCAN failure condition**: when clusters overlap and density varies smoothly, `eps` becomes extremely sensitive and no automatic rule pins it down.


In [ ]:
# --- To merge the micro-clusters into the two intended segments, eps must be far larger ---
nbrsHotMore = NearestNeighbors(n_neighbors=60)
nbrsHotMore.fit(scaled_df)
distancesHotMore = np.sort(nbrsHotMore.kneighbors(scaled_df)[0][:, -1], axis=0)

In [ ]:
axHotMore = pd.Series(distancesHotMore).plot(kind='line', figsize=(8, 5))
axHotMore.set(ylabel='Distance to 30th nearest neighbor', xlabel='Point (sorted)')

In [ ]:
kneeHotMore = KneeLocator(range(len(distancesHotMore)),distancesHotMore,S=1,curve='convex',direction='increasing',online=False)
kneeHotMore.plot_knee() # 绘制曲线与拐点位置
distancesHotMore[kneeHotMore.elbow]

In [ ]:
goodDB = DBSCAN(eps=0.463, min_samples=60, metric='euclidean')
goodLabels = goodDB.fit_predict(scaled_df)
pd.Series(goodLabels).value_counts().sort_index()

With `eps ≈ 0.5` — near the top of the distance curve — DBSCAN finally returns the two intended segments. But notice how fragile the choice is: a small change in `eps` swings the result between 2 and 9 clusters. This sensitivity motivates methods that do not need a single global `eps`:
- **HDBSCAN** handles varying density without one fixed `eps`
- **GMM** models overlapping, not necessarily spherical, segments with soft assignments


In [ ]:
from sklearn.datasets import make_moons

In [ ]:
X,y = make_moons(n_samples=200,noise=0.05,random_state=0)
X[:5,:]
y[:5]

In [ ]:
pd.DataFrame(X).plot(kind='scatter',x=0,y=1,figsize=(12,6),c='steelblue')

### K-Means Clustering


In [ ]:
km = KMeans(n_clusters=2,random_state=0)
y_km = km.fit_predict(X)
ax_km = pd.DataFrame(X[y_km==0,:]).plot(kind='scatter',x=0,y=1,figsize=(12,6),c='lightblue',edgecolor='black',marker='o',s=40,label='cluster 1')
pd.DataFrame(X[y_km==1,:]).plot(kind='scatter',x=0,y=1,ax=ax_km,c='red',edgecolor='black',marker='s',s=40,label='cluster 2')
ax_km.set(title='K-means clustering')

### Agglomerative Clustering


In [ ]:
ac = AgglomerativeClustering(n_clusters=2,metric='euclidean',linkage='complete')
y_ac = ac.fit_predict(X)
ax_ac = pd.DataFrame(X[y_ac==0,:]).plot(kind='scatter',x=0,y=1,figsize=(12,6),c='lightblue',edgecolor='black',marker='o',s=40,label='cluster 1')
pd.DataFrame(X[y_ac==1,:]).plot(kind='scatter',x=0,y=1,ax=ax_ac,c='red',edgecolor='black',marker='s',s=40,label='cluster 2')
ax_ac.set(title='Agglomerative clustering')

### Density-Based Clustering


In [ ]:
db = DBSCAN(eps=0.2,min_samples=5,metric='euclidean')
y_db = db.fit_predict(X)
ax_db = pd.DataFrame(X[y_db==0,:]).plot(kind='scatter',x=0,y=1,figsize=(12,6),c='lightblue',edgecolor='black',marker='o',s=40,label='cluster 1')
pd.DataFrame(X[y_db==1,:]).plot(kind='scatter',x=0,y=1,ax=ax_db,c='red',edgecolor='black',marker='s',s=40,label='cluster 2')
ax_db.set(title='DBSCAN clustering')

## HDBSCAN — Hierarchical Density-Based Clustering

<div class="alert alert-warning">
    <strong>Why not just DBSCAN?</strong> DBSCAN uses a single global <code>eps</code>. Dense and sparse clusters need different radii — one radius cannot fit both (recall the varying-density example above).
</div>


### The HDBSCAN Idea

<div class="alert alert-info">
    <strong>HDBSCAN</strong> (Hierarchical DBSCAN) solves the main weakness of DBSCAN: it can find clusters of <strong>varying densities</strong> and is much more robust to hyperparameters.
</div>

#### The 5-Step Intuition:
1. **Transform the space**: Adjust distances based on local density (Mutual Reachability Distance).
2. **Build a Minimum Spanning Tree**: Connect points such that the total distance is minimized.
3. **Build a hierarchy**: Construct a dendrogram of the connected components.
4. **Condense the tree**: Prune branches that have fewer than `min_cluster_size` points.
5. **Extract the clusters**: Keep the "survivors"—clusters that persist over the widest range of density levels.

<center><img src="./img/clustering/hdbscan_process.png" width="80%"></center>

<div class="alert alert-warning">
    <strong>The Synthesis: Why this fits the "No Single Radius" Idea</strong>
</div>

In DBSCAN, you pick one fixed radius ($\epsilon$). This fails if Cluster A is very tight and Cluster B is sparse.

HDBSCAN's 5-step process is consistent with the **no single radius** philosophy because:
- **It doesn't choose; it explores**: The hierarchy (Step 3) looks at *every* possible radius level simultaneously.
- **It measures persistence**: Instead of asking "is this group denser than X?", it asks "how long does this group stay together as we vary the density threshold?"
- **It's adaptive**: By extracting "survivors" (Step 5) based on stability, it can keep a dense group at one level and a sparse group at another—something impossible with a single global threshold.

#### Mathematical Foundation: Mutual Reachability

To handle varying densities, HDBSCAN "pushes" sparse points further away using the **Mutual Reachability Distance**:

$$d_{\text{mre}}(a, b) = \max\{core_k(a), core_k(b), d(a, b)\}$$

Where:
- $d(a, b)$ is the standard Euclidean distance.
- $core_k(x)$ is the distance to the $k$-th nearest neighbor (local density).

<div class="alert alert-success">
    <strong>Why this matters:</strong> Dense points stay close, but sparse points (low density) are effectively "pushed" into the noise, making clusters cleaner and more distinct.
</div>

### How to Set the HDBSCAN Parameters


```python
HDBSCAN(min_cluster_size=5, min_samples=None, metric='euclidean', cluster_selection_method='eom', prediction_data=False)
```
- `min_cluster_size`: Minimum size of clusters; smaller values allow for more clusters, larger values yield fewer clusters. Rule of thumb: start at $2 \times n_{features}$, then raise it for larger datasets
- `min_samples`: Controls how conservative the clustering is; higher values make the algorithm more robust to noise. Default equal to `min_cluster_size` — usually a good starting point
- `metric`: Distance metric to use (default is Euclidean).
- `cluster_selection_method`: Method to select clusters from the hierarchy; 'eom' (excess of mass) is the default and works well in most cases.

#### Practical workflow
1. Start with `min_cluster_size = 2 \times n_features` (or a business-defined segment size)
2. Leave `min_samples` at its default (equal to `min_cluster_size`)
3. Plot the condensed tree / cluster hierarchy to see how many stable clusters exist
4. Tune `min_cluster_size` up and down; merge leftovers with `cluster_selection_epsilon`
5. Unlike K-Means, HDBSCAN needs **no number of clusters K** — it discovers them from the data


#### HDBSCAN on Hotel Guests


In [ ]:
# --- HDBSCAN clustering of hotel guests ---
# Requires: !pip install hdbscan
from hdbscan import HDBSCAN

hdb = HDBSCAN(min_cluster_size=50, metric='euclidean')
hdb_labels = hdb.fit_predict(scaled_df)

pd.Series(hdb_labels).value_counts().sort_index()

In [ ]:
# --- Visualize the HDBSCAN segments ---
plt.figure(figsize=(8, 6))
scatter = plt.scatter(scaled_df.iloc[:, 0], scaled_df.iloc[:, 1],
                      c=hdb_labels, cmap="tab10", s=40, alpha=0.8)
plt.colorbar(scatter, label='Segment (-1 = noise)')
plt.xlabel('Stay Duration (standardized)')
plt.ylabel('Daily Spend (standardized)')
plt.title('HDBSCAN Segmentation of Hotel Guests')
plt.show()

In [ ]:
# --- HDBSCAN vs DBSCAN on data with varying density ---
from sklearn.datasets import make_blobs

# One dense blob, one dense blob, one sparse blob
Xv, _ = make_blobs(n_samples=[300, 300, 300], centers=[[0, 0], [5, 0], [0, 5]],
                   cluster_std=[0.3, 0.3, 1.5], random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].scatter(Xv[:, 0], Xv[:, 1], c='lightblue', s=15)
axes[0].set_title('Raw data (varying density)')

dbv = DBSCAN(eps=0.5, min_samples=20).fit_predict(Xv)
axes[1].scatter(Xv[:, 0], Xv[:, 1], c=dbv, cmap="tab10", s=15)
axes[1].set_title('DBSCAN: sparse blob lost or merged')

hdv = HDBSCAN(min_cluster_size=50, metric="euclidean").fit_predict(Xv)
axes[2].scatter(Xv[:, 0], Xv[:, 1], c=hdv, cmap="tab10", s=15)
axes[2].set_title('HDBSCAN: all three densities found')
plt.show()

<div class="alert alert-warning">
    <strong>Note:</strong> like DBSCAN, HDBSCAN labels unusual points as noise (<code>-1</code>) and has <strong>no <code>predict</code></strong> for brand-new points — re-run it on the growing dataset periodically.
</div>


## Gaussian Mixture Models (GMM)

<div class="alert alert-info">
    <strong>Beyond Hard Circles:</strong> While K-Means assumes every cluster is a perfect circle, GMM realizes that real customer groups come in all shapes and sizes (ellipses).
</div>

#### The Core Intuition
GMM assumes that the data was generated by a mixture of $K$ different Normal (Gaussian) distributions. 

$$
p(\mathbf{x}) = \sum_{k=1}^{K} \pi_k \; \mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)
$$

<center><img src="./img/clustering/kmeans_vs_gmm_shapes.png" width="70%"></center>

- **Flexibility**: By learning the **Covariance matrix** ($\boldsymbol{\Sigma}_k$), GMM can model clusters that are stretched, rotated, or squashed.

#### The Generative Story

To understand GMM, imagine a machine that "paints" your customer data:
1. **Pick a group**: First, it picks one of $K$ buckets based on the probability $\pi_k$ (e.g., "60% chance it's a Budget guest").
2. **Spray the points**: Then, it "sprays" points around the center $\boldsymbol{\mu}_k$ according to the shape $\boldsymbol{\Sigma}_k$.

<center><img src="./img/clustering/gmm_generative_process.png" width="70%"></center>

<div class="alert alert-success">
    <strong>Business Insight:</strong> GMM doesn't just find groups; it estimates the <strong>likelihood</strong> of every guest belonging to every group. This is the power of <strong>Soft Assignment</strong>.
</div>

### How GMM Learns: Expectation-Maximization (EM)

The algorithm iteratively solves a "chicken-and-egg" problem:
1. **E-step (Expectation)**: "If we knew the Gaussians, which points would belong to which?" $\rightarrow$ Calculate probabilities $P(k \mid \mathbf{x})$.
2. **M-step (Maximization)**: "If we knew the point memberships, where should the Gaussians be?" $\rightarrow$ Update $\boldsymbol{\mu}_k$ and $\boldsymbol{\Sigma}_k$.

#### Model Selection: Finding the right $K$
We use the **Bayesian Information Criterion (BIC)** to find the optimal number of segments:
$$\text{BIC} = k \ln(n) - 2 \ln(\widehat{L})$$
- It rewards the model for explaining the data well (Likelihood $L$).
- It penalizes the model for being too complex (adding too many clusters $k$).

### GMM in Python

```python
GaussianMixture(n_components=k, random_state=42)
```
- `n_components`: Number of mixture components (clusters)
- `random_state`: Seed for reproducibility
- `covariance_type`: Type of covariance parameters to use ('full', 'tied', 'diag', 'spherical')
- `max_iter`: Maximum number of iterations for the EM algorithm
- `tol`: Convergence threshold for EM algorithm
- `init_params`: Method for initialization ('kmeans', 'random', 'kmeans++')

### How to Set the GMM Parameters


#### `n_components` — the only essential choice

The number of Gaussians (clusters) $K$. Unlike DBSCAN/HDBSCAN, GMM **does not discover $K$ automatically** — you must set it.
- Too small → distinct segments get merged
- Too large → extra components split dense groups or absorb noise
- Choose it with **BIC / AIC** (look for an elbow), then sanity-check with silhouette scores and segment sizes against the business context
- The demo below uses `n_components=3`


#### `covariance_type` — the shape GMM is allowed to draw
- `'full'` (default): each segment has its own tilted ellipse → most flexible, most parameters
- `'tied'`: all segments share one ellipse shape and orientation
- `'diag'`: axis-aligned ellipses
- `'spherical'`: circles — reduces to a "soft K-Means"

More flexible shapes fit more complex data but need more data to estimate reliably (risk of overfitting).


#### Initialization & reproducibility
- `init_params='kmeans'` (default): start from K-Means centers (fast, usually good); `'random'` starts randomly
- `n_init`: number of restarts; the best one (lowest BIC/AIC) is kept. Raise it (e.g., 5-10) if results vary between runs
- `random_state`: fix it so every run (and every student) gets identical results


#### Convergence & numerical safety
- `max_iter` (default 100): EM iterations; if `gmm.n_iter_` hits `max_iter`, the model did not converge — raise it
- `tol` (default `1e-3`): EM stops when the improvement per iteration is smaller than this
- `reg_covar` (default `1e-6`): tiny value added to the covariance diagonal to keep it invertible. Increase it (e.g., `1e-4` or `1e-2`) if you get a *"singular covariance matrix"* warning or error

#### Practical workflow
1. Compare BIC/AIC across $K$ (elbow) → shortlist 2-3 values
2. Check silhouette + segment sizes; pick the $K$ that also makes business sense
3. Start with `covariance_type='full'`; fall back to `'diag'` if overfitting
4. Fix `random_state=42`; set `n_init=5` if unstable
5. Raise `reg_covar` if covariance warnings appear


#### GMM on Hotel Guests

We reuse the standardized hotel features loaded in the HDBSCAN demo above: stay duration and daily spend.


In [ ]:
# --- Choose the number of components with BIC (clean synthetic example) ---
from sklearn.mixture import GaussianMixture
from sklearn.datasets import make_blobs

Ks = range(1, 8)
bics = [GaussianMixture(n_components=k, random_state=42).fit(scaled_df).bic(scaled_df) for k in Ks]

pd.Series(bics, index=Ks).plot(kind='line', marker='o', figsize=(8, 5))
plt.title('BIC by number of components (lower is better)')
plt.xlabel('K'); plt.ylabel('BIC')
plt.show()
print(f"Best K by BIC: {1 + int(np.argmin(bics))}")

<div class="alert alert-warning">
    <strong>Real-data caveat:</strong> on the actual hotel data, BIC keeps improving as $K$ grows (extra components absorb noise). Do not pick $K$ from BIC alone — combine it with silhouette scores (Clustering I) and business interpretability. We use $K=3$ below because it gives interpretable, actionable segments.
</div>


In [ ]:
# --- Fit GMM and get soft + hard assignments ---
gmm = GaussianMixture(n_components=3, random_state=42)
gmm_labels = gmm.fit_predict(scaled_df)     # hard labels (argmax of responsibilities)
gmm_proba = gmm.predict_proba(scaled_df)    # soft assignment

print("Cluster sizes (hard labels):", np.bincount(gmm_labels))
print("Soft probabilities for the first 5 guests (3 segments):")
print(pd.DataFrame(gmm_proba[:5], columns=[f"segment {k}" for k in range(3)]).round(2))

In [ ]:
# --- Visualize GMM segments and profile them on the original scale ---
plt.figure(figsize=(8, 6))
scatter = plt.scatter(scaled_df.iloc[:, 0], scaled_df.iloc[:, 1],
                      c=gmm_labels, cmap="Set1", s=40, alpha=0.8)
plt.colorbar(scatter, label='Segment')
plt.xlabel('Stay Duration (standardized)')
plt.ylabel('Daily Spend (standardized)')
plt.title('GMM Segmentation of Hotel Guests (K=3)')
plt.show()

profile = hotdf.groupby(gmm_labels)[["Stay_Duration_Days", "Daily_Spend_ADR"]].mean()
print(profile.round(2))

<div class="alert alert-success">
    <strong>Read it:</strong> GMM draws <strong>elliptical, tilted</strong> segment boundaries (not circles) — it can separate guests that K-Means cannot, and <code>predict_proba</code> tells you <em>how sure</em> the model is about each guest.
</div>


## Putting It Together — Hotel Guest Analytics

| Guest type | Found by | Suggested action |
|------------|----------|------------------|
| High spend + long stay | All methods | Loyalty program, upgrade offers |
| Irregular / niche segment | DBSCAN, HDBSCAN | Targeted niche campaigns |
| Unusual behavior (label -1) | DBSCAN, HDBSCAN | Manual review — VIP or abuse? |
| Overlapping / uncertain guest | GMM soft probability | Hybrid offers, nurture |


### Method Comparison

| Aspect | K-Means | DBSCAN | HDBSCAN | GMM |
|--------|:---:|:---:|:---:|:---:|
| Cluster shape | Spherical | Any | Any | Elliptical |
| Noise handling | ✗ | ✓ (-1) | ✓ (-1) | ✗ (low probability) |
| Soft assignment | ✗ | ✗ | ✗ | ✓ |
| Needs K | ✓ | ✗ | ✗ | ✓ (via BIC) |
| `eps` tuning | — | ✓ | ✗ | — |
| New-point prediction | ✓ | ✗ | ✗ | ✓ |


### Key Takeaways

| Aspect | Summary |
|--------|---------|
| **When** | Irregular shapes + noise → DBSCAN / HDBSCAN; overlapping + uncertainty → GMM |
| **HDBSCAN** | Modern upgrade of DBSCAN; no `eps`; handles varying density |
| **GMM** | Probabilistic soft clustering via EM; confidence per guest |
| **Workflow** | Standardize features → choose method by shape/noise/uncertainty → validate with silhouette or DBI (Clustering I) |
